In [6]:
import pytesseract
import os
import re
from PIL import Image

def get_most_recent_file(directory):
    
    # List all files    
    files = [os.path.join(directory, f) for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith('.png')]

    # Get the most recent
    most_recent_file = max(files, key=os.path.getctime)
    
    return most_recent_file


# Get the screenshot
directory_path = "/Users/konstantin/Documents"
recent_file = get_most_recent_file(directory_path)
print("Found the screenshot file", recent_file)

# Load image
image = Image.open(recent_file)

# Set language to Russian ('rus')
text = pytesseract.image_to_string(image, lang='rus+eng')

UNINFORMATIVE_RE = re.compile(
    r'[\d.,]+\s?(₽|р|Р|P|руб\.?)|Срок годности',
    re.IGNORECASE
)

def extract_product_title(lines):
    pattern = r'^\d+$'
    return sorted([line.strip() for line in lines if not UNINFORMATIVE_RE.search(line) and len(line)>5 and not re.fullmatch(pattern, line.replace(' ','').strip())])

# Example usage
titles = extract_product_title(text.split('\n'))

for i,x in enumerate(titles):
    print(i, ': ', repr(x))


Found the screenshot file /Users/konstantin/Documents/Screenshot 2025-07-17 at 20.24.04.png
0 :  176? 2292 . 7Ог
1 :  1792. 60О0г
2 :  2392. ИОг
3 :  Картофель запечёный с ветчиной и сыром
4 :  Молоко 2,5% Домик в деревне
5 :  Рис жареный самбал с овощами
6 :  Ролл-фри крабовый
7 :  Салат из кальмаров и моркови по-корейски
8 :  Сосиски из индейки Из Лавки
9 :  Сырочки
10 :  Филе бедра индейки с перцем Sabroso Monte Индилайт
11 :  Яблоки Гренни
12 :  молоко
13 :  №250717-117-6484


In [2]:
# Manual merging
for pair in [(2, 6)]:
    joined = ' '.join([titles[pair[0]], titles[pair[1]]])
    print("Merged: ",joined)
    titles[pair[0]] = joined
    titles[pair[1]] = ''

titles = [x for x in titles if x != '']

for i,x in enumerate(titles):
    print(i, ': ', x)

Merged:  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
0 :  Батончик протеиновый ProteinRex кокос
1 :  Дыня нарезанная кубиками «Из Лавки»
2 :  Мороженое протеиновое Bombbar фисташковое без сахара в вафельном стаканчике
3 :  Продукт творожный «Даниссимо» с сочным киви 5,5%
4 :  Салат оливье с курицей «Из Лавки»
5 :  Сэндвич стунцом и маринованным луком «Из Лавки»


### Вариант 2 (рабочий): из HTML читаем

In [112]:
from bs4 import BeautifulSoup
import json
import re

html_file = "/Users/konstantin/Documents/Заказ оформлен.html"
with open(html_file, "r", encoding="utf-8") as file:
    soup = BeautifulSoup(file, "html.parser")
    
raw_text = ''
for script in soup.find_all("script"):
    if len(script.text) > 100 and script.text.startswith("window.__REACT_QUERY_STATE__"):
        raw_text = script.text[29:]

# чиcтим чтобы влезло в JSON
match = re.search(r'({.*})', raw_text, re.DOTALL)
if not match:
    raise ValueError("No JSON-like block found in the file.")
json_like_text = match.group(1)
cleaned_json_text = (
    json_like_text
    .replace("undefined", "null")  # JS -> JSON
    .replace("True", "true")
    .replace("False", "false")
)

try:
    data = json.loads(cleaned_json_text)
except json.JSONDecodeError as e:
    raise ValueError(f"JSON parsing error: {e}")

# json[json.find("Рис")-50:json.find("Рис")+100]
titles = [x['name'] for x in data['queries'][1]['state']['data']['calculation']['items']]

for i, title in enumerate(titles, 1):
    print(f"{i}. {title}")


1. Грудка куриная запечённая 2 шт. «Из Лавки»
2. Сельдь с картофелем и маринованным красным луком «Йуми»
3. Пирожное песочное Суфле в шоколаде «Из Лавки»
4. Энергетический напиток Red Bull
5. Рис жареный самбал с овощами «Старик и море»
6. Форель радужная филе-кусок «Из Лавки» с кожей, с овощами и травами замороженная
7. Медальоны из филе тунца с брокколи «Из Лавки»
8. Десерт глазированный Fit Kit фисташковый крем
9. Фриттата с курицей и брокколи «Из Лавки»
10. Шакшука с варёным куриным яйцом «Шук»
11. Печенье протеиновое Chikalab с фисташковым суфле без сахара
12. Масса творожная Exponenta со вкусом вишня
13. Молоко 2,5% «Домик в деревне» ультрапастеризованное


## Справочник калорийности

In [114]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# Define scope
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

# Load credentials
creds = ServiceAccountCredentials.from_json_keyfile_name('credentials.json', scope)
client = gspread.authorize(creds)

# Open Google Sheet by name or URL
spreadsheet = client.open("Калории")

# Select worksheet/tab by name
worksheet = spreadsheet.worksheet("Reference")

# Get all values as list of rows
data = worksheet.get_all_values()

reference = pd.DataFrame(data[1:], columns=data[0])

reference


,продукт,вес,ккал на 100г,ккал,б на 100г,ж на 100г,у на 100г,б на порцию,ж на порцию,у на порцию
0,Аджапсандал,2,75.5,151,1.4,6.3,9.3,2.8,12.6,18.6
1,Баклажаны Пармиджано с соусом Болоньезе «Йуми»,2,173,346,5.9,13.1,8,11.8,26.2,16
2,Батончик вафельный Snaq Fabriq с молочно-орехо...,0.2,394,78.8,8,27,17,1.6,5.4,3.4
3,Батончик протеиновый Bombbar фисташковый пломбир,0.6,314,188.4,33,11,3.7,19.8,6.6,2.22
4,Блины с сёмгой «Шоколадница»,1.5,187,280.5,11.4,8.9,15.3,17.1,13.35,22.95
...,...,...,...,...,...,...,...,...,...,...
85,Энергетический напиток Red Bull,2.5,46,115,0,0,11,0,0,27.5
86,Яблоки Гренни,6,59,354,0.2,0.4,15.3,1.2,2.4,91.8
87,Яблоки Гренни Смит Отборные «Собрано в саду» 4 шт,8,47,376,0.4,4,9.7,3.2,32,77.6
88,Яблоки Малинка,6,59,354,0.2,0.4,15.3,1.2,2.4,91.8


In [115]:
from datetime import datetime
import numpy as np

# Match products with reference records
update = reference[reference['продукт'].isin(titles)]

print("Not found: ", set(list(titles)).difference(set(reference['продукт'])))

# Insert date columns
current_date = datetime.today().strftime('%Y%m%d')
# current_date = '20250717'; print("SETTING manual date")
update.insert(0, 'дата', current_date)
update.insert(len(update.columns), 'активность', np.nan)

# Use Formulas instead of constants
update.loc[:,'ккал'] = 0.0
update.loc[:,'б на порцию'] = 0.0
update.loc[:,'ж на порцию'] = 0.0
update.loc[:,'у на порцию'] = 0.0

# Update types
update.loc[:,'вес'] = update['вес'].astype(float)
update.loc[:,'ккал на 100г'] = update['ккал на 100г'].astype(float)
update.loc[:,'б на 100г'] = update['б на 100г'].astype(float)
update.loc[:,'ж на 100г'] = update['ж на 100г'].astype(float)
update.loc[:,'у на 100г'] = update['у на 100г'].astype(float)

# Compute totals
totals = update.iloc[:,2:].sum()
totals_row = pd.DataFrame([[current_date, 'Total'] + totals.tolist()], columns=update.columns)
update = pd.concat([update, totals_row], ignore_index=True)

# Append the blank row
blank_row = {col:np.nan for col in update.columns}
blank_row['дата']=current_date
update = pd.concat([update, pd.DataFrame([blank_row])], ignore_index=True)

update

Not found:  set()


,дата,продукт,вес,ккал на 100г,ккал,б на 100г,ж на 100г,у на 100г,б на порцию,ж на порцию,у на порцию,активность
0,20250720,Грудка куриная запечённая 2 шт. «Из Лавки»,1.4,167.0,0.0,31.0,4.7,0.0,0.0,0.0,0.0,NaN
1,20250720,Десерт глазированный Fit Kit фисташковый крем,0.5,280.0,0.0,26.0,8.0,24.0,0.0,0.0,0.0,NaN
2,20250720,Масса творожная Exponenta со вкусом вишня,2.5,70.0,0.0,16.0,0.0,1.5,0.0,0.0,0.0,NaN
3,20250720,Медальоны из филе тунца с брокколи «Из Лавки»,2.0,69.0,0.0,6.5,4.3,1.0,0.0,0.0,0.0,NaN
4,20250720,"Молоко 2,5% «Домик в деревне» ультрапастеризов...",9.5,53.0,0.0,2.9,2.5,4.7,0.0,0.0,0.0,NaN
5,20250720,Печенье протеиновое Chikalab с фисташковым суф...,0.55,339.0,0.0,22.0,17.0,10.2,0.0,0.0,0.0,NaN
6,20250720,Пирожное песочное Суфле в шоколаде «Из Лавки»,0.6,374.0,0.0,3.0,18.0,50.0,0.0,0.0,0.0,NaN
7,20250720,Рис жареный самбал с овощами «Старик и море»,1.5,153.0,0.0,3.7,1.8,30.4,0.0,0.0,0.0,NaN
8,20250720,Сельдь с картофелем и маринованным красным лук...,2.0,168.0,0.0,6.2,10.5,12.1,0.0,0.0,0.0,NaN
9,20250720,"Форель радужная филе-кусок «Из Лавки» с кожей,...",3.0,92.0,0.0,7.8,5.5,2.7,0.0,0.0,0.0,NaN


In [116]:
from gspread_dataframe import get_as_dataframe, set_with_dataframe

upd_worksheet = spreadsheet.worksheet("2025 New")

existing = get_as_dataframe(upd_worksheet, evaluate_formulas=True, header=0)
first_row = len(existing) + 5

# Step 5: Append new data
set_with_dataframe(
    upd_worksheet, 
    update, 
    row=first_row,
    col=1,
    include_column_header=True)

In [104]:
print(update.head())

       дата                                            продукт  вес  \
0  20250715       Картофель запечёный с ветчиной и сыром Mates  2.2   
1  20250715  Молоко 2,5% «Домик в деревне» ультрапастеризов...  9.5   
2  20250715       Рис жареный самбал с овощами «Старик и море»  1.5   
3  20250715                                  Ролл-фри крабовый  1.8   
4  20250715  Салат из кальмаров командорских и моркови по-к...  1.1   

  ккал на 100г ккал б на 100г ж на 100г у на 100г б на порцию ж на порцию  \
0        133.0  0.0       8.3       5.6      12.3         0.0         0.0   
1         53.0  0.0       2.9       2.5       4.7         0.0         0.0   
2        153.0  0.0       3.7       1.8      30.4         0.0         0.0   
3        236.0  0.0       6.2       7.3      36.4         0.0         0.0   
4        175.0  0.0       6.0      13.0       9.0         0.0         0.0   

  у на порцию  активность  
0         0.0         NaN  
1         0.0         NaN  
2         0.0         NaN 

In [83]:
!pip3.13 install gspread_dataframe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [gspread]1/12 [gspread]uth]]s]ib]
